In [4]:
import nltk
import random
import math
from collections import Counter, defaultdict

nltk.download('treebank', quiet=True)
from nltk.corpus import treebank

sentences = [
    [(word.lower(), tag) for word, tag in sent]
    for sent in treebank.tagged_sents()
]

random.seed(42)
random.shuffle(sentences)

split = int(0.8 * len(sentences))
train = sentences[:split]
test = sentences[split:]

START = "<START>"
UNK = "<UNK>"
alpha = 0.1

trans = defaultdict(Counter)
emit = defaultdict(Counter)
tag_count = Counter()
word_count = Counter()
tags = set()

for sent in train:
    for word, tag in sent:
        word_count[word] += 1

for sent in train:
    prev = START
    for word, tag in sent:
        if word_count[word] <= 1:
            word = UNK

        trans[prev][tag] += 1
        emit[tag][word] += 1
        tag_count[tag] += 1
        tags.add(tag)
        prev = tag

tags = sorted(tags)
vocab = {word for counts in emit.values() for word in counts}
vocab.add(UNK)


In [5]:
K = len(tags)
V = len(vocab)

def transition(prev, tag):
    return math.log(
        (trans[prev][tag] + alpha) /
        (sum(trans[prev].values()) + alpha * K)
    )

def emission(tag, word):
    if word not in vocab:
        word = UNK

    return math.log(
        (emit[tag][word] + alpha) /
        (tag_count[tag] + alpha * V)
    )

def viterbi(words):
    n = len(words)
    dp = [{} for _ in range(n)]
    back = [{} for _ in range(n)]

    if n == 0:
        return []

    for tag in tags:
        dp[0][tag] = transition(START, tag) + emission(tag, words[0])

    for i in range(1, n):
        for tag in tags:
            scores = [
                dp[i - 1][prev] + transition(prev, tag)
                + emission(tag, words[i])
                for prev in tags
            ]

            best = max(range(K), key=lambda j: scores[j])
            dp[i][tag] = scores[best]
            back[i][tag] = tags[best]

    tag = max(dp[-1], key=dp[-1].get)
    result = [tag]

    for i in range(n - 1, 0, -1):
        tag = back[i][tag]
        result.append(tag)

    return result[::-1]

correct = 0
total = 0

for sent in test:
    words = [word for word, tag in sent]
    actual = [tag for word, tag in sent]
    predicted = viterbi(words)

    for a, p in zip(actual, predicted):
        total += 1
        if a == p:
            correct += 1

print("Accuracy:", round(correct / total * 100, 2), "%")

sentence = "The market is growing rapidly".lower().split()
predicted = viterbi(sentence)

for word, tag in zip(sentence, predicted):
    print(word, ":", tag)

Accuracy: 90.47 %
the : DT
market : NN
is : VBZ
growing : VBG
rapidly : RB
